# DataMate V5 trên Kaggle

Chọn GPU T4. Thêm dataset chứa DataMate_V5 và dataset Training. Kaggle có thể hiển thị ZIP hoặc thư mục đã giải nén. Điền hai đường dẫn trong cell đầu nếu có nhiều kết quả. Nếu có dataset cache llama.cpp T4 từ V4.2, thêm vào Input để dùng lại.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
PROJECT = None  # E.g. Path('/kaggle/input/datasets/hawndang/datamate-v5/datamate_v5')
TRAIN = None    # E.g. Path('/kaggle/input/datasets/hawndang/train1/train')

def one(paths, label):
    files = sorted(set(paths))
    if len(files) != 1:
        raise RuntimeError(f'Expected one {label}, found: {files}. Set the path explicitly.')
    return files[0]

if PROJECT is None:
    dirs = [p.parent for p in INPUT.rglob('kaggle_setup.py') if (p.parent / 'datamate').is_dir()]
    project_zips = [p for p in INPUT.rglob('*.zip') if p.stem.casefold().startswith('datamate_v5')]
    PROJECT = one(dirs or project_zips, 'project')
else:
    PROJECT = Path(PROJECT)
if TRAIN is None:
    dirs = [p.parent for p in INPUT.rglob('answers.jsonl')
            if all((p.parent / name).exists() for name in ('questions.jsonl', 'rulebook.md', 'data'))]
    TRAIN = one(dirs or INPUT.rglob('train*.zip'), 'Training split')
else:
    TRAIN = Path(TRAIN)
for label, path in (('PROJECT', PROJECT), ('TRAIN', TRAIN)):
    if not path.exists():
        raise FileNotFoundError(f'{label}: {path}')
PUBLIC = next(INPUT.rglob('public_test*.zip'), None)
MODEL = next(INPUT.rglob('Qwen3.5-9B-Q4_K_M.gguf'), None)
SERVER = None  # Override only if you have a known compatible executable.
ROOT = WORK / 'datamate_v5'
if PROJECT.is_dir():
    shutil.copytree(PROJECT, ROOT, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(PROJECT) as z:
        z.extractall(WORK)

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, check=True)

run('-m', 'pip', 'install', '-r', 'requirements.txt')
print('Code:', ROOT)
print('Training:', TRAIN)


In [ ]:
# Prepare GGUF and llama-server. Each benchmark owns and can restart the process.
previous_server = globals().get('server')
if previous_server is not None and previous_server.poll() is None:
    previous_server.terminate()
    try:
        previous_server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_server.kill()
        previous_server.wait()

runtime_file = WORK / 'datamate_runtime_paths.json'
args = ['kaggle_setup.py', '--prepare', runtime_file]
if MODEL:
    args += ['--model', MODEL]
if SERVER:
    args += ['--server', SERVER]
run(*args)
runtime = json.loads(runtime_file.read_text())
print('Runtime ready. The benchmark loads Qwen when a question needs it.')

RUNS = WORK / 'datamate_benchmarks'

def benchmark(phase, *extra):
    run('benchmark.py', phase, '--data', TRAIN, '--model', runtime['model'],
        '--server', runtime['server'], '--output', RUNS / phase, *extra)


In [ ]:
# Smoke test 10 câu. Bật thử logprob để kiểm tra cache llama-server trên Kaggle.
benchmark('smoke', '--log-confidence')


In [ ]:
# Compare all 80 original V4.2 IDs, baseline +13.
benchmark('regression', '--log-confidence')


In [ ]:
# Optional loader audit for Public Test.
if PUBLIC:
    run('-m', 'datamate', 'audit', '--data', PUBLIC, '--output', WORK/'v5_public_audit')


In [ ]:
# Holdout 40 câu/biz. Confidence lấy từ token logprob nếu runtime hỗ trợ.
HOLDOUT_BIZ = []  # Điền các biz Training team chưa dùng để debug.
if HOLDOUT_BIZ:
    choices = [part for biz in HOLDOUT_BIZ for part in ('--business', biz)]
    benchmark('holdout', '--per-business', 40, '--log-confidence', *choices)


In [ ]:
# Chọn hai nhóm biz Training độc lập, chưa dùng để debug hoặc xem gold.
CALIBRATION_BIZ = []
VALIDATION_BIZ = []
if CALIBRATION_BIZ and VALIDATION_BIZ:
    assert not set(CALIBRATION_BIZ) & set(VALIDATION_BIZ), "Hai nhóm phải tách theo biz."
    for name, businesses in [('calibration', CALIBRATION_BIZ), ('validation', VALIDATION_BIZ)]:
        args = [part for biz in businesses for part in ('--business', biz)]
        run('benchmark.py', 'holdout', '--data', TRAIN, '--model', runtime['model'],
            '--server', runtime['server'], '--output', RUNS / name,
            '--per-business', 40, '--log-confidence', *args)
    run('threshold.py', '--data', TRAIN, '--calibration', RUNS/'calibration',
        '--validation', RUNS/'validation', '--output', RUNS/'threshold_validation')


In [ ]:
# Download each benchmark as soon as it has usable output.
for phase in ('smoke', 'regression', 'holdout', 'calibration', 'validation', 'threshold_validation'):
    folder = RUNS / phase
    if (folder / 'predictions.csv').is_file():
        print(shutil.make_archive(str(WORK / f'datamate_{phase}'), 'zip', folder))

if all((RUNS / phase / 'metrics.json').is_file() for phase in ('smoke', 'regression', 'holdout')):
    print(shutil.make_archive(str(WORK / 'datamate_benchmarks'), 'zip', RUNS))
